# Exp 3-v8 🏁 — TF-IDF + 전처리 + 핸드크래프트 피처 + **W&B Sweep**

## 왜 Sweep이 필요한가?
- v7 전처리 + 핸드크래프트 피처 추가로 **입력 공간이 완전히 바뀜**
- 기존 best config (lr=5.59e-05, hidden=256)는 raw TF-IDF 기준으로 찾은 것
- 새 입력 (30006차원, 전처리된 특성)에 맞는 최적 config 재탐색 필요

## 학습 곡선 분석 (v7)
- Epoch 36에서 Dev best (0.6907) → 그 이후 하락
- lr이 너무 낮아서 수렴이 느리고, 더 높은 hidden_size 시도 여지 있음

## 예상 성능: **70%+** 🎯

In [1]:
!pip install datasets wandb scikit-learn -q

In [2]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from scipy.sparse import hstack, csr_matrix
import numpy as np, copy, re, wandb
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [3]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}|Dev:{len(dev_data)}|Test:{len(test_data)}|Classes:{output_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

Train:31232|Dev:5205|Test:5205|Classes:3


In [4]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.replace('`', "'")
    text = text.replace('****', ' bad ')
    text = text.replace('***', ' bad ')
    text = re.sub(r'!{3,}', ' verymuch ! ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'cannot', text)
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"'re", ' are', text)
    text = re.sub(r"'ve", ' have', text)
    text = re.sub(r"'ll", ' will', text)
    text = re.sub(r"'d", ' would', text)
    text = re.sub(r"'m", ' am', text)
    slangs = [
        (r'\bidk\b','i do not know'),(r'\bur\b','your'),
        (r'\bnaw\b','no'),(r'\bgonna\b','going to'),
        (r'\bwanna\b','want to'),(r'\blol\b','laughing'),
        (r'\bomg\b','oh my god'),(r'\bwtf\b','what the'),
        (r'\bugh\b','disgusting'),(r'\btho\b','though'),
        (r'\bkinda\b','kind of'),(r'\bcuz\b','because'),
        (r'\bsoo+\b','so'),(r'\bthx\b','thanks'),
        (r'\byep\b','yes'),(r'\byup\b','yes'),
        (r'\bnope\b','no'),(r'\btbh\b','to be honest'),
        (r'\bimo\b','in my opinion'),
    ]
    for pat,rep in slangs:
        text=re.sub(pat,rep,text)
    return text

def extract_handcraft(texts):
    features=[]
    for text in texts:
        t=str(text)
        tl=t.lower()
        words=t.split()
        features.append([
            min(t.count('!'),5),
            min(t.count('?'),5),
            sum(1 for w in words if w.isupper() and len(w)>1),
            min(len(words),50),
            int(bool(re.search(r'http\S+',tl))),
            int(any(e in tl for e in [':)',':(',':d',':/','haha','hehe','lmao'])),
        ])
    return np.array(features,dtype=np.float32)

# 전처리 + 핸드크래프트 피처 빌드 (Sweep에서 재사용)
vectorizer=TfidfVectorizer(max_features=30000,preprocessor=preprocess_text,min_df=2)
vectorizer.fit(train_data['text'])

def build_features(data_split):
    tfidf_mat=vectorizer.transform(data_split['text'])
    hc_mat=csr_matrix(extract_handcraft(data_split['text']))
    return torch.FloatTensor(hstack([tfidf_mat,hc_mat]).toarray()).to(device)

train_t=build_features(train_data)
dev_t=build_features(dev_data)
test_t=build_features(test_data)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
input_size=train_t.shape[1]
print(f'입력 크기:{input_size}')

입력 크기:11663


In [5]:
class MLP(nn.Module):
    def __init__(self,i,h,o,d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [6]:
sweep_config={
    'method':'bayes',
    'metric':{'name':'best_dev_accuracy','goal':'maximize'},
    'parameters':{
        'hidden_size':{'values':[256,512,1000]},
        'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-3},
        'dropout':{'values':[0.1,0.2,0.3,0.4]},
        'weight_decay':{'values':[0,1e-5,1e-4]},
        'batch_size':{'values':[128,256]},
        'num_epochs':{'values':[30,50,70]},
    }
}
sweep_id=wandb.sweep(sweep_config,project='nlp-hw1')
print(f'Sweep ID:{sweep_id}')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: 8oy7hj3j
Sweep URL: https://wandb.ai/imeanseo_/nlp-hw1/sweeps/8oy7hj3j
Sweep ID:8oy7hj3j


In [7]:
def train_sweep():
    run=wandb.init()
    cfg=run.config
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=MLP(input_size,cfg.hidden_size,output_size,cfg.dropout).to(device)
    opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    lfn=nn.CrossEntropyLoss()
    best_dev,best_state=0,None
    for epoch in range(cfg.num_epochs):
        model.train()
        total_loss=0
        for i in range(0,len(train_t),cfg.batch_size):
            bd=train_t[i:i+cfg.batch_size]
            bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
            loss=lfn(model(bd),bl)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss+=loss.item()
        model.eval()
        with torch.no_grad():
            da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
        if da>best_dev:
            best_dev,best_state=da,copy.deepcopy(model.state_dict())
        wandb.log({'epoch':epoch+1,'train_loss':total_loss/max(1,len(train_t)//cfg.batch_size),
                   'dev_accuracy':da,'best_dev_accuracy':best_dev})
    model.load_state_dict(best_state)
    with torch.no_grad():
        test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
    wandb.log({'test_accuracy':test_acc})
    print(f'[Exp3-v8] Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')
    wandb.finish()

wandb.agent(sweep_id,train_sweep,count=15)

wandb: Agent Starting Run: riz5ap7u with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.4
wandb: 	hidden_size: 256
wandb: 	learning_rate: 0.00012478164672055273
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aileen02-ko (imeanseo_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[Exp3-v8] Dev:0.6882|Test:68.68%


best_dev_accuracy,▁▃▅▇▇▇████████████████████████
dev_accuracy,▁▃▅▇▇▇█▇█████████████████▇▇███
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,█▇▆▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68818
dev_accuracy,0.67358
epoch,30
test_accuracy,0.68684
train_loss,0.72282


wandb: Agent Starting Run: 8rrf0yz6 with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.3
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 0.0009087056356018596
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp3-v8] Dev:0.6738|Test:67.74%


...

[Exp3-v8] Dev:0.6970|Test:69.15%


best_dev_accuracy,▁▃▅▆▆███████████████████████████████████
dev_accuracy,▁▂▅▆▇▇▇████████████████████████████████▇
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
test_accuracy,▁
train_loss,█▇▆▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.69702
dev_accuracy,0.67666
epoch,70
test_accuracy,0.69145
train_loss,0.68978


wandb: Agent Starting Run: ual98nql with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 0.000395474227558229
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp3-v8] Dev:0.6890|Test:68.90%


best_dev_accuracy,▁▅████████████████████████████
dev_accuracy,▂▆█▇▇▅▇▅▄▆▃▃▅▇▃▂▂▆▃▁▂▅▆▄▃▅▆▆▅▂
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,█▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68895
dev_accuracy,0.64111
epoch,30
test_accuracy,0.68895
train_loss,0.72988


In [8]:
# Best config로 최종 학습 + 저장
api=wandb.Api()
sweep=api.sweep(f'imeanseo_/nlp-hw1/{sweep_id}')
best_run=sorted(sweep.runs,key=lambda r:r.summary.get('best_dev_accuracy',0),reverse=True)[0]
cfg=best_run.config
print(f'Best config: {cfg}')

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model=MLP(input_size,cfg['hidden_size'],output_size,cfg['dropout']).to(device)
opt=optim.Adam(model.parameters(),lr=cfg['learning_rate'],weight_decay=cfg['weight_decay'])
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
print('🏁 최종 학습...')
for epoch in range(cfg['num_epochs']):
    model.train()
    for i in range(0,len(train_t),cfg['batch_size']):
        bd=train_t[i:i+cfg['batch_size']]
        bl=torch.tensor(train_labels[i:i+cfg['batch_size']],device=device)
        loss=lfn(model(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(model.state_dict())
        print(f'✨ Epoch {epoch+1}|Dev:{da:.4f}|NEW BEST!')
model.load_state_dict(best_state)
torch.save(best_state,'best_model_exp3_v8_final.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장: best_model_exp3_v8_final.pt')
print(f'📊 Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')
print(f'\n🎯 목표 70%: {"달성! 🎉" if test_acc>=0.70 else f"{test_acc*100:.2f}% (기준 68.61% 대비 {(test_acc-0.6861)*100:+.2f}%p)"}')

Best config: {'dropout': 0.4, 'batch_size': 256, 'num_epochs': 70, 'hidden_size': 1000, 'weight_decay': 0.0001, 'learning_rate': 3.649238347809796e-05}
🏁 최종 학습...
✨ Epoch 1|Dev:0.4110|NEW BEST!
✨ Epoch 2|Dev:0.4359|NEW BEST!
✨ Epoch 3|Dev:0.4574|NEW BEST!
✨ Epoch 4|Dev:0.4726|NEW BEST!
✨ Epoch 5|Dev:0.5036|NEW BEST!
✨ Epoch 6|Dev:0.5285|NEW BEST!
✨ Epoch 7|Dev:0.5562|NEW BEST!
✨ Epoch 8|Dev:0.5967|NEW BEST!
✨ Epoch 9|Dev:0.6250|NEW BEST!
✨ Epoch 10|Dev:0.6340|NEW BEST!
✨ Epoch 11|Dev:0.6461|NEW BEST!
✨ Epoch 12|Dev:0.6565|NEW BEST!
✨ Epoch 13|Dev:0.6567|NEW BEST!
✨ Epoch 14|Dev:0.6630|NEW BEST!
✨ Epoch 15|Dev:0.6701|NEW BEST!
✨ Epoch 16|Dev:0.6707|NEW BEST!
✨ Epoch 17|Dev:0.6728|NEW BEST!
✨ Epoch 18|Dev:0.6772|NEW BEST!
✨ Epoch 19|Dev:0.6805|NEW BEST!
✨ Epoch 20|Dev:0.6824|NEW BEST!
✨ Epoch 21|Dev:0.6870|NEW BEST!
✨ Epoch 23|Dev:0.6901|NEW BEST!
✨ Epoch 24|Dev:0.6934|NEW BEST!
✨ Epoch 26|Dev:0.6941|NEW BEST!
✨ Epoch 27|Dev:0.6953|NEW BEST!
✨ Epoch 28|Dev:0.6955|NEW BEST!
✨ Epoch 29|Dev

In [9]:
from google.colab import files
files.download('best_model_exp3_v8_final.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>